# Feature engineering — Prévision du trafic à t+15 minutes

Projet : **Traffic Prediction & Adaptive Signal Control Agent**, dataset **PEMS08**.

Chaque observation est espacée de 5 minutes : les cibles à **t+15 minutes** correspondent aux mesures du même capteur **3 timesteps après t**. Les observations présentes et passées servent aux features ; les valeurs futures sont réservées aux cibles. Aucun modèle n'est entraîné ici.


In [1]:
# Importer les outils de manipulation des données et de gestion des chemins.
from pathlib import Path
import pandas as pd

# Accepter une exécution depuis notebooks ou depuis la racine du projet.
chemin_source = Path("../data/raw/traffic_dataset.csv")
if not chemin_source.exists():
    chemin_source = Path("data/raw/traffic_dataset.csv")

# Charger le CSV sans modifier le fichier source.
df = pd.read_csv(chemin_source)
print("Shape du dataset source :", df.shape)
df.head()


Shape du dataset source : (3035520, 5)


,timestep,location,flow,occupy,speed
0,1,0,133.0,0.0603,65.8
1,1,1,210.0,0.0589,69.6
2,1,2,124.0,0.0358,65.8
3,1,3,145.0,0.0416,69.6
4,1,4,206.0,0.0493,69.4


## Ordre temporel et vérifications

Un décalage de lignes représente une durée exacte seulement si les timesteps sont consécutifs pour chaque capteur. Les anomalies source doivent être examinées plutôt que supprimées silencieusement.


In [2]:
# Vérifier les colonnes, les valeurs manquantes et l'unicité des observations.
colonnes_source = ["timestep", "location", "flow", "speed", "occupy"]
assert set(colonnes_source).issubset(df.columns), "Des colonnes nécessaires sont absentes."
assert not df[colonnes_source].isna().any().any(), "Des valeurs source sont manquantes."
assert not df.duplicated(["location", "timestep"]).any(), "Des observations sont dupliquées."
assert (df["timestep"] % 1 == 0).all(), "Les timesteps doivent être entiers."

# Trier par capteur, puis par temps ; remettre l'index des lignes à zéro.
df = df.sort_values(["location", "timestep"]).reset_index(drop=True)

# Vérifier la continuité temporelle pour que shift(3) corresponde bien à 15 minutes.
ecarts = df.groupby("location")["timestep"].diff()
assert ecarts.dropna().eq(1).all(), "Des timesteps manquent : vérifier la grille temporelle."

# Exiger assez de mesures pour l'historique maximal et la cible future.
assert df.groupby("location").size().gt(15).all(), "Historique insuffisant pour un capteur."
print("Nombre de capteurs :", df["location"].nunique())
df.head()


Nombre de capteurs : 170


,timestep,location,flow,occupy,speed
0,1,0,133.0,0.0603,65.8
1,2,0,114.0,0.0532,66.9
2,3,0,140.0,0.0622,66.8
3,4,0,106.0,0.0452,68.9
4,5,0,117.0,0.0574,67.4


## Variables temporelles

L'origine est commune aux capteurs : le premier timestep correspond au **jour 0, à 00:00 par convention**. Cette heure relative ne garantit pas l'heure réelle de mesure sans métadonnées complémentaires.

**day_index** commence à 0 et **slot_in_day** va de 0 à 287. Aucune date calendrier ni aucun jour de la semaine n'est inventé.


In [3]:
# Une journée contient 24 * 60 / 5 = 288 intervalles.
pas_minutes = 5
intervalles_par_jour = 288

# Utiliser une origine globale pour partager le même repère entre capteurs.
origine_timestep = df["timestep"].min()
temps_relatif = (df["timestep"] - origine_timestep).astype("int64")

# Calculer le numéro du jour et la position dans la journée.
df["day_index"] = temps_relatif // intervalles_par_jour
df["slot_in_day"] = temps_relatif % intervalles_par_jour

# Convertir chaque intervalle en heure et minute.
minutes_dans_jour = df["slot_in_day"] * pas_minutes
df["hour"] = minutes_dans_jour // 60
df["minute"] = minutes_dans_jour % 60

# Préparer 288 libellés HH:MM puis les associer aux observations.
# time_label facilite la lecture sans créer de fausses dates.
libelles_horaires = {
    slot: f"{slot * pas_minutes // 60:02d}:{slot * pas_minutes % 60:02d}"
    for slot in range(intervalles_par_jour)
}
df["time_label"] = df["slot_in_day"].map(libelles_horaires)
df[["location", "timestep", "day_index", "slot_in_day", "hour", "minute", "time_label"]].head()


,location,timestep,day_index,slot_in_day,hour,minute,time_label
0,0,1,0,0,0,0,00:00
1,0,2,0,1,0,5,00:05
2,0,3,0,2,0,10,00:10
3,0,4,0,3,0,15,00:15
4,0,5,0,4,0,20,00:20


## Lags par capteur

Les retards 1, 2, 3, 6 et 12 correspondent à **5, 10, 15, 30 et 60 minutes dans le passé**. Le regroupement par location empêche de récupérer une observation d'un autre capteur.


In [4]:
# Construire les cinq retards pour chacune des trois mesures.
variables_trafic = ["flow", "speed", "occupy"]
retards = [1, 2, 3, 6, 12]
colonnes_lags = []

# Un shift positif récupère le passé du même capteur.
# Les lignes sans historique suffisant restent manquantes à cette étape.
for variable in variables_trafic:
    for retard in retards:
        nom_colonne = f"{variable}_lag_{retard}"
        df[nom_colonne] = df.groupby("location")[variable].shift(retard)
        colonnes_lags.append(nom_colonne)

df[["location", "timestep", "flow"] + colonnes_lags[:5]].head(15)


,location,timestep,flow,flow_lag_1,flow_lag_2,flow_lag_3,flow_lag_6,flow_lag_12
0,0,1,133.0,NaN,NaN,NaN,NaN,NaN
1,0,2,114.0,133.0,NaN,NaN,NaN,NaN
2,0,3,140.0,114.0,133.0,NaN,NaN,NaN
3,0,4,106.0,140.0,114.0,133.0,NaN,NaN
4,0,5,117.0,106.0,140.0,114.0,NaN,NaN
5,0,6,110.0,117.0,106.0,140.0,NaN,NaN
6,0,7,130.0,110.0,117.0,106.0,133.0,NaN
7,0,8,112.0,130.0,110.0,117.0,114.0,NaN
8,0,9,115.0,112.0,130.0,110.0,140.0,NaN
9,0,10,93.0,115.0,112.0,130.0,106.0,NaN


## Moyennes glissantes strictement passées

À t, la fenêtre de taille 3 utilise **t−3, t−2 et t−1**. Les tailles 3, 6 et 12 résument les **15, 30 et 60 minutes précédentes**.

Le décalage préalable exclut la mesure courante. Chaque fenêtre doit être complète : aucune valeur future ni valeur d'un autre capteur n'est utilisée.


In [5]:
# Calculer les moyennes glissantes indépendamment pour chaque capteur.
fenetres = [3, 6, 12]
colonnes_rolling = []

# transform conserve l'alignement avec les lignes du DataFrame.
# shift(1) exclut t et min_periods exige une fenêtre complète.
for variable in variables_trafic:
    for fenetre in fenetres:
        nom_colonne = f"{variable}_rolling_mean_{fenetre}"
        df[nom_colonne] = df.groupby("location")[variable].transform(
            lambda serie: serie.shift(1).rolling(
                window=fenetre, min_periods=fenetre
            ).mean()
        )
        colonnes_rolling.append(nom_colonne)

df[["location", "timestep"] + colonnes_rolling].head(15)


,location,timestep,flow_rolling_mean_3,flow_rolling_mean_6,flow_rolling_mean_12,speed_rolling_mean_3,speed_rolling_mean_6,speed_rolling_mean_12,occupy_rolling_mean_3,occupy_rolling_mean_6,occupy_rolling_mean_12
0,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0,4,129.000000,NaN,NaN,66.500000,NaN,NaN,0.058567,NaN,NaN
4,0,5,120.000000,NaN,NaN,67.533333,NaN,NaN,0.053533,NaN,NaN
5,0,6,121.000000,NaN,NaN,67.700000,NaN,NaN,0.054933,NaN,NaN
6,0,7,111.000000,120.000000,NaN,67.266667,66.883333,NaN,0.052367,0.055467,NaN
7,0,8,119.000000,119.500000,NaN,66.433333,66.983333,NaN,0.056167,0.054850,NaN
8,0,9,117.333333,119.166667,NaN,64.966667,66.333333,NaN,0.057233,0.056083,NaN
9,0,10,119.000000,115.000000,NaN,65.566667,66.416667,NaN,0.055033,0.053700,NaN


## Cibles à t+15 minutes

Un décalage négatif de 3 récupère les mesures futures du même capteur, uniquement pour les cibles. Les trois dernières observations de chaque capteur n'ont pas de cible disponible.


In [6]:
# Trois observations représentent 3 * 5 = 15 minutes.
horizon = 3
colonnes_targets = []

# Le décalage négatif est réservé aux cibles et reste interne à chaque capteur.
for variable in variables_trafic:
    nom_colonne = f"target_{variable}_t15"
    df[nom_colonne] = df.groupby("location")[variable].shift(-horizon)
    colonnes_targets.append(nom_colonne)

# Vérifier que le timestep de chaque cible disponible est exactement t + 3.
timestep_cible = df.groupby("location")["timestep"].shift(-horizon)
cible_disponible = timestep_cible.notna()
assert timestep_cible[cible_disponible].eq(
    df.loc[cible_disponible, "timestep"] + horizon
).all()

df[["location", "timestep"] + variables_trafic + colonnes_targets].head()


,location,timestep,flow,speed,occupy,target_flow_t15,target_speed_t15,target_occupy_t15
0,0,1,133.0,65.8,0.0603,106.0,68.9,0.0452
1,0,2,114.0,66.9,0.0532,117.0,67.4,0.0574
2,0,3,140.0,66.8,0.0622,110.0,65.5,0.0545
3,0,4,106.0,68.9,0.0452,130.0,66.4,0.0566
4,0,5,117.0,67.4,0.0574,112.0,63.0,0.0606


## Nettoyage des lignes sans historique ou cible

Nous retirons uniquement les **12 premières** observations de chaque capteur et les **3 dernières**. Les moyennes glissantes ne nécessitent pas de suppression supplémentaire, leur historique maximal étant également de 12 observations. Aucune imputation ni aucun filtrage des valeurs de trafic n'est appliqué.


In [7]:
# Enregistrer la shape avant le nettoyage.
shape_avant = df.shape
nombre_capteurs = df["location"].nunique()
taille_historique = max(max(retards), max(fenetres))

# Identifier uniquement les NaN créés dans les lags et les cibles.
colonnes_validite = colonnes_lags + colonnes_targets
lignes_invalides = df[colonnes_validite].isna().any(axis=1)

# Vérifier que les moyennes glissantes n'ont pas d'autres NaN.
assert not df.loc[~lignes_invalides, colonnes_rolling].isna().any().any()

# Vérifier que les lignes invalides sont exactement les bords de chaque série.
position_capteur = df.groupby("location").cumcount()
taille_capteur = df.groupby("location")["timestep"].transform("size")
bords_attendus = (
    (position_capteur < taille_historique)
    | (position_capteur >= taille_capteur - horizon)
)
assert lignes_invalides.equals(bords_attendus)

# Supprimer uniquement les lignes sans historique ou cible.
df_features = df.dropna(subset=colonnes_validite).reset_index(drop=True)

# Comparer les shapes et vérifier le nombre de suppressions.
print("Shape avant nettoyage :", shape_avant)
print("Shape après nettoyage :", df_features.shape)
print("Lignes supprimées :", shape_avant[0] - len(df_features))
assert shape_avant[0] - len(df_features) == nombre_capteurs * (taille_historique + horizon)

# Vérifier les NaN dans toutes les colonnes du résultat final.
print("NaN restants par colonne :")
print(df_features.isna().sum().to_string())
assert not df_features.isna().any().any(), "Des NaN inattendus subsistent."


Shape avant nettoyage : (3035520, 37)
Shape après nettoyage : (3032970, 37)
Lignes supprimées : 2550
NaN restants par colonne :
timestep                  0
location                  0
flow                      0
occupy                    0
speed                     0
day_index                 0
slot_in_day               0
hour                      0
minute                    0
time_label                0
flow_lag_1                0
flow_lag_2                0
flow_lag_3                0
flow_lag_6                0
flow_lag_12               0
speed_lag_1               0
speed_lag_2               0
speed_lag_3               0
speed_lag_6               0
speed_lag_12              0
occupy_lag_1              0
occupy_lag_2              0
occupy_lag_3              0
occupy_lag_6              0
occupy_lag_12             0
flow_rolling_mean_3       0
flow_rolling_mean_6       0
flow_rolling_mean_12      0
speed_rolling_mean_3      0
speed_rolling_mean_6      0
speed_rolling_mean_12     0
occu

In [8]:
# Montrer plusieurs lignes de trois capteurs avec les features et les cibles.
# Le tableau complet reste disponible dans df_features.
colonnes_exemples = [
    "location", "timestep", "day_index", "time_label",
    "flow", "flow_lag_1", "flow_lag_12", "flow_rolling_mean_3",
    "speed", "speed_lag_1", "speed_rolling_mean_6",
    "occupy", "occupy_lag_1", "occupy_rolling_mean_12"
] + colonnes_targets
capteurs_exemples = df_features["location"].drop_duplicates().head(3)
exemples = (
    df_features.loc[df_features["location"].isin(capteurs_exemples)]
    .groupby("location")
    .head(3)
)
exemples[colonnes_exemples]


,location,timestep,day_index,time_label,flow,flow_lag_1,flow_lag_12,flow_rolling_mean_3,speed,speed_lag_1,speed_rolling_mean_6,occupy,occupy_lag_1,occupy_rolling_mean_12,target_flow_t15,target_speed_t15,target_occupy_t15
0,0,13,0,01:00,87.0,93.0,133.0,92.000000,66.7,64.3,65.433333,0.0451,0.0496,0.053108,96.0,64.5,0.0524
1,0,14,0,01:05,78.0,87.0,114.0,90.000000,63.4,66.7,65.483333,0.0481,0.0451,0.051842,82.0,64.6,0.0457
2,0,15,0,01:10,98.0,78.0,140.0,86.000000,64.3,63.4,65.550000,0.0534,0.0481,0.051417,85.0,63.3,0.0496
17841,1,13,0,01:00,135.0,139.0,210.0,137.333333,68.3,68.9,67.650000,0.0426,0.0403,0.046700,116.0,70.7,0.0349
17842,1,14,0,01:05,136.0,135.0,185.0,137.000000,71.3,68.3,68.233333,0.0374,0.0426,0.045342,127.0,66.7,0.0482
17843,1,15,0,01:10,114.0,136.0,171.0,136.666667,72.1,71.3,69.016667,0.0325,0.0374,0.043875,131.0,68.3,0.0399
35682,2,13,0,01:00,76.0,97.0,124.0,98.000000,63.8,64.4,64.133333,0.0285,0.0311,0.032050,78.0,64.7,0.0289
35683,2,14,0,01:05,85.0,76.0,119.0,89.666667,64.2,63.8,64.333333,0.0303,0.0285,0.031442,71.0,64.3,0.0276
35684,2,15,0,01:10,87.0,85.0,107.0,86.000000,64.0,64.2,64.466667,0.0290,0.0303,0.031142,81.0,63.6,0.0302


## Résultat et prochaine étape

**df_features** contient les mesures courantes, les variables temporelles, **15 lags**, **9 moyennes glissantes** et **3 cibles à t+15 minutes**, sans mélange entre capteurs. Aucun modèle n'est entraîné.

La future évaluation devra utiliser un découpage chronologique commun aux capteurs et exclure les lignes d'entraînement dont les cibles franchissent la frontière de validation. Les colonnes **target_*** ne devront jamais servir de features. L'utilisation des mesures courantes suppose une prédiction après réception des mesures à t.


## Séparation temporelle : train, validation et test

Un split aléatoire mélangerait passé et futur : le modèle pourrait être entraîné sur des observations postérieures à celles utilisées pour l'évaluation. Des fenêtres proches partagent aussi des mesures et une cible à t+15 peut se retrouver parmi les observations d'entraînement. Cette dépendance rendrait l'évaluation trop optimiste pour une prévision future.

Nous découpons donc les **timesteps uniques disponibles dans df_features**, triés chronologiquement : **70 % pour train**, **15 % pour validation**, puis **le reste pour test**. Les tailles des deux premières périodes sont arrondies à l'entier inférieur. Les frontières sont communes à tous les capteurs, sans découpage aléatoire.

Pour empêcher les cibles de franchir une frontière, nous retirons ensuite les trois derniers timesteps de train et de validation : leur cible à t+3 appartient à la période suivante. Les proportions finales diffèrent donc légèrement de 70/15/15. Les cibles du test sont déjà disponibles grâce au nettoyage précédent.

Les lags du début d'une période peuvent utiliser l'historique de la période précédente : ces mesures étaient déjà connues au moment de la prédiction. Cela correspond à une évaluation où une nouvelle prévision est produite à chaque timestep avec les observations reçues jusqu'à t.


In [9]:
# Déterminer les frontières sur les timesteps uniques, et non sur les lignes.
# Ainsi, un même timestep est affecté à la même période pour tous les capteurs.
timesteps_disponibles = sorted(df_features["timestep"].unique())
nombre_timesteps = len(timesteps_disponibles)
nombre_train = int(nombre_timesteps * 0.70)
nombre_validation = int(nombre_timesteps * 0.15)
nombre_test = nombre_timesteps - nombre_train - nombre_validation

# Vérifier que chaque période possède assez de timesteps pour l'horizon futur.
assert min(nombre_train, nombre_validation, nombre_test) > horizon

# La première observation de validation et de test définit chaque frontière.
debut_validation = timesteps_disponibles[nombre_train]
debut_test = timesteps_disponibles[nombre_train + nombre_validation]

print("Timesteps disponibles :", nombre_timesteps)
print("Répartition avant purge :", nombre_train, nombre_validation, nombre_test)
print("Début de validation :", debut_validation)
print("Début de test :", debut_test)


Timesteps disponibles : 17841
Répartition avant purge : 12488 2676 2677
Début de validation : 12501
Début de test : 15177


In [10]:
# Affecter les observations aux trois périodes chronologiques communes.
temps = df_features["timestep"]
periode_train = temps < debut_validation
periode_validation = (temps >= debut_validation) & (temps < debut_test)
periode_test = temps >= debut_test

# Une cible doit rester strictement avant le début de la période suivante.
# horizon vaut 3 : les dernières lignes de train et validation sont donc purgées.
cibles_dans_train = temps + horizon < debut_validation
cibles_dans_validation = temps + horizon < debut_test

# Conserver les features et les cibles dans trois DataFrames distincts.
train = df_features.loc[periode_train & cibles_dans_train].copy()
validation = df_features.loc[periode_validation & cibles_dans_validation].copy()
test = df_features.loc[periode_test].copy()

# Comptabiliser explicitement les lignes retirées aux frontières.
lignes_purgees_train = int((periode_train & ~cibles_dans_train).sum())
lignes_purgees_validation = int((periode_validation & ~cibles_dans_validation).sum())
print("Lignes purgées à la fin de train :", lignes_purgees_train)
print("Lignes purgées à la fin de validation :", lignes_purgees_validation)


Lignes purgées à la fin de train : 510
Lignes purgées à la fin de validation : 510


In [ ]:
# Résumer les plages réellement conservées après la purge des frontières.
ensembles = {"Train": train, "Validation": validation, "Test": test}
resume_split = pd.DataFrame([
    {
        "Ensemble": nom,
        "Timestep minimum": ensemble["timestep"].min(),
        "Timestep maximum": ensemble["timestep"].max(),
        "Timesteps distincts": ensemble["timestep"].nunique(),
        "Nombre de lignes": len(ensemble),
        "Nombre de capteurs": ensemble["location"].nunique(),
    }
    for nom, ensemble in ensembles.items()
]).set_index("Ensemble")

# Afficher les plages temporelles et les effectifs des trois ensembles.
print(resume_split.to_string())

# Vérifier l'ordre chronologique et l'absence de cibles traversant les frontières.
assert train["timestep"].max() + horizon < validation["timestep"].min()
assert validation["timestep"].max() + horizon < test["timestep"].min()

# Vérifier que toutes les lignes sont soit conservées, soit explicitement purgées.
assert (
    len(train) + len(validation) + len(test)
    + lignes_purgees_train + lignes_purgees_validation
) == len(df_features)

# Vérifier que chaque période contient les mêmes capteurs et les mêmes timesteps.
capteurs_attendus = set(df_features["location"].unique())
for nom, ensemble in ensembles.items():
    assert set(ensemble["location"].unique()) == capteurs_attendus
    assert ensemble.groupby("timestep")["location"].nunique().eq(len(capteurs_attendus)).all()
    assert not ensemble.isna().any().any(), f"Des NaN subsistent dans {nom}."


            Timestep minimum  Timestep maximum  Timesteps distincts  Nombre de lignes  Nombre de capteurs
Ensemble                                                                                                 
Train                     13             12497                12485           2122450                 170
Validation             12501             15173                 2673            454410                 170
Test                   15177             17853                 2677            455090                 170


: 

### Ensembles prêts pour la suite

**train**, **validation** et **test** conservent les mêmes colonnes et couvrent des périodes successives communes aux capteurs. Le tableau indique les plages après purge ; les petits intervalles exclus empêchent les cibles d'empiéter sur l'ensemble suivant.

Aucun modèle n'est entraîné. Lors de l'étape suivante, les colonnes **target_*** seront séparées des features et les éventuelles transformations apprises, comme la normalisation, seront ajustées uniquement sur train.
